# View past MAAP DPS jobs

The DPS **Jobs** panel shows a job's *status badge*, but the thing you usually need is
its **log** — and a green `successful` badge does **not** mean the job did what you
wanted. A `run.sh` that catches its own error and exits 0 (e.g. `dps/probe/run.sh`,
which reports an IAM denial as a *result* rather than crashing) is reported as
successful. **Always read `_stdout.txt`.**

This notebook lists past jobs, inspects one, and prints its stdout/stderr.

**Kernel:** `Python 3 (disasters_dps)`.

---

## API gotchas this notebook exists to encode

- **Use the `snake_case` methods.** `maap.getJobStatus()` / `maap.getJobResult()` are
  no-arg shims — calling `maap.getJobStatus(job_id)` raises
  `TypeError: takes 1 positional argument but 2 were given`. The real API is
  `maap.get_job_status(job_id)` / `get_job_result(job_id)` / `get_job_metrics(job_id)`
  and `maap.list_jobs(...)` (keyword-only).
- **They return `requests.Response`, not dicts.** Call `.json()` on them.
- **Job outputs live in a MAAP workspace bucket.** The Disasters hub's ambient identity
  (`disasters-prod`, account `515966502221`) cannot read it — you need short-lived
  credentials from `maap.aws.workspace_bucket_credentials()`. Same mechanism
  `shared_utils/staging_upload.py` uses to publish to `nasa-disasters-staging`.

Related: [`dps/delete_algorithm.ipynb`](delete_algorithm.ipynb) (undeploy an algorithm),
[`docs/DPS.md`](../docs/DPS.md).

## 1. Connect

In [ ]:
import json
import re

import boto3
import requests
from maap.maap import MAAP

maap = MAAP()


def as_json(resp):
    """Unwrap what the get_job_* helpers return.

    maap-py returns a ``requests.Response`` from ``get_job_status`` /
    ``get_job_result`` / ``get_job_metrics``. Older/newer builds have returned a
    plain dict or a string, so normalize instead of assuming. On a non-JSON body
    (MAAP sometimes answers with XML) the raw text is returned so you can still
    read it rather than getting a JSONDecodeError.
    """
    if isinstance(resp, (dict, list)):
        return resp
    if isinstance(resp, str):
        return resp
    body = getattr(resp, "text", None)
    status = getattr(resp, "status_code", None)
    if status is not None and status >= 400:
        print(f"HTTP {status}")
    try:
        return resp.json()
    except Exception:
        return body


def show(obj, limit=4000):
    """Pretty-print a dict/list, or echo text, truncated."""
    s = obj if isinstance(obj, str) else json.dumps(obj, indent=2, default=str)
    print(s[:limit] + ("\n... [truncated]" if len(s) > limit else ""))


print("Connected as:", maap.profile.account_info().get("username", "<unknown>"))

## 2. List recent jobs

`list_jobs` is **keyword-only**. Useful filters:

| kwarg | example | note |
|---|---|---|
| `process_id` | `"disasters-umbra-process"` | the registered algorithm name |
| `status` | `"failed"` | also `succeeded` / `running` (MAAP's spelling, not the badge's) |
| `tag` | `"test"` | the free-text tag you typed on the Submit form |
| `page_size` | `25` | results per page |
| `offset` | `25` | page through with `page_size` |

Leave everything unset for "my most recent jobs".

In [ ]:
# --- filters (None = don't filter) ---
PROCESS_ID = None      # e.g. "disasters-iam-probe"
STATUS = None          # e.g. "failed"
TAG = None             # e.g. "test"
PAGE_SIZE = 25
OFFSET = 0

kwargs = {"page_size": PAGE_SIZE, "offset": OFFSET}
if PROCESS_ID:
    kwargs["process_id"] = PROCESS_ID
if STATUS:
    kwargs["status"] = STATUS
if TAG:
    kwargs["tag"] = TAG

jobs = as_json(maap.list_jobs(**kwargs))

# The payload shape varies by maap-py build: sometimes a bare list, sometimes
# {"jobs": [...]}. Normalize rather than index blindly.
if isinstance(jobs, dict):
    rows = jobs.get("jobs") or jobs.get("results") or []
else:
    rows = jobs if isinstance(jobs, list) else []

print(f"{len(rows)} job(s)\n")


def pick(d, *names, default="-"):
    """First present, non-empty key among ``names`` (payload keys drift by build)."""
    for n in names:
        v = d.get(n)
        if v not in (None, "", []):
            return v
    return default


for j in rows:
    if not isinstance(j, dict):
        print(j)
        continue
    jid = pick(j, "jobID", "job_id", "id", "payload_id")
    proc = pick(j, "processID", "process_id", "algorithm", "job_type")
    st = pick(j, "status", "job_status", "state")
    when = pick(j, "queued", "time_queued", "created", "submitted")
    print(f"{str(jid):40s} {str(st):12s} {str(when):26s} {proc}")

# Full shape of one entry, if you need a field the table above doesn't show.
if rows:
    print("\nKEYS:", sorted(rows[0].keys()) if isinstance(rows[0], dict) else type(rows[0]))

## 3. Inspect one job

Paste a job ID (the UUID in the Jobs panel, e.g. `cd1488a9-…`).

In [ ]:
JOB_ID = "cd1488a9-c595-44d8-b1e3-9d2275cbb020"   # <-- edit me

print("=== STATUS ===")
show(as_json(maap.get_job_status(JOB_ID)))

print("\n=== RESULT ===")
result = as_json(maap.get_job_result(JOB_ID))
show(result)

Metrics (runtime, memory, the worker instance) — handy when a job OOMs and you need to
raise `ram_min` in the algorithm config.

In [ ]:
show(as_json(maap.get_job_metrics(JOB_ID)))

## 4. Read the job log

This is the part the Jobs panel makes awkward. DPS writes `_stdout.txt` and
`_stderr.txt` into the job's output directory.

> **Your `run.sh` output is in `_stderr.txt`, not `_stdout.txt`.**
> The job is executed by `cwltool`, which relays the container's console output to
> **stderr**. `_stdout.txt` holds only the DPS wrapper's own chatter (fetching the
> CWL from `maap-ops-registry`, echoing the parsed inputs, `=== Starting Stage Out
> Process ===`) — typically under 10 lines. A job whose `run.sh` printed 200 lines
> still shows a nearly empty `_stdout.txt`. **Read both; default to `_stderr.txt`.**

The result payload's shape is not stable across maap-py versions, so rather than
indexing a guessed key we **scrape every `s3://` / `https://` string out of it** and
work from those.

In [ ]:
def find_locations(obj):
    """Recursively collect every s3:// or http(s):// string in a nested payload."""
    found = []

    def walk(o):
        if isinstance(o, str):
            found.extend(re.findall(r"(?:s3|https?)://[^\s\"'<>,\]}]+", o))
        elif isinstance(o, dict):
            for v in o.values():
                walk(v)
        elif isinstance(o, (list, tuple)):
            for v in o:
                walk(v)

    walk(obj)
    # de-dup, preserve order
    return list(dict.fromkeys(found))


locations = find_locations(result)
for loc in locations:
    print(loc)

if not locations:
    print("No URLs in the result payload — the job may still be running, or it "
          "failed before producing outputs. Check the STATUS output above.")

In [ ]:
def parse_s3_location(uri):
    """Return ``(bucket, prefix)`` from ANY of the location forms MAAP returns.

    The result payload does NOT give you a clean ``s3://bucket/key``. Observed forms
    for a single job, all pointing at the same place:

      s3://s3-us-west-2.amazonaws.com:80/maap-ops-workspace/kdl0040/dps_output/...
        ^ ENDPOINT-prefixed. Naive ``uri[5:].split("/", 1)`` yields bucket
          "s3-us-west-2.amazonaws.com:80" and a key that starts with the real
          bucket name -> NoSuchBucket, or worse, a silent empty listing.
      http://maap-ops-workspace.s3-website-us-west-2.amazonaws.com/kdl0040/...
      https://s3.console.aws.amazon.com/s3/buckets/maap-ops-workspace/kdl0040/...?region=...

    Ordered most-specific first.
    """
    u = uri.strip()

    # AWS console deep-link
    m = re.match(r"https?://s3\.console\.aws\.amazon\.com/s3/buckets/([^/?]+)/?([^?]*)", u)
    if m:
        return m.group(1), m.group(2).rstrip("/")

    # virtual-hosted / website endpoint: <bucket>.s3[-website]-<region>.amazonaws.com
    m = re.match(r"https?://([^./]+)\.s3[.-](?:website[.-])?[a-z0-9-]+\.amazonaws\.com/?([^?]*)", u)
    if m:
        return m.group(1), m.group(2).rstrip("/")

    # endpoint-prefixed s3 URI: s3://<endpoint>[:port]/<bucket>/<key>
    m = re.match(r"s3://[^/]*amazonaws\.com(?::\d+)?/([^/]+)/?(.*)", u)
    if m:
        return m.group(1), m.group(2).rstrip("/")

    # plain s3://<bucket>/<key>
    m = re.match(r"s3://([^/]+)/?(.*)", u)
    if m:
        return m.group(1), m.group(2).rstrip("/")

    raise ValueError(f"unrecognized S3 location: {uri}")


# Sanity-check against the three forms MAAP actually returned for one job.
for _u in [
    "s3://s3-us-west-2.amazonaws.com:80/maap-ops-workspace/kdl0040/dps_output/x/dev/1",
    "http://maap-ops-workspace.s3-website-us-west-2.amazonaws.com/kdl0040/dps_output/x/dev/1",
    "https://s3.console.aws.amazon.com/s3/buckets/maap-ops-workspace/kdl0040/dps_output/x/dev/1/?region=us-east-1",
    "s3://maap-ops-workspace/kdl0040/dps_output/x/dev/1",
]:
    assert parse_s3_location(_u) == (
        "maap-ops-workspace", "kdl0040/dps_output/x/dev/1"
    ), (_u, parse_s3_location(_u))
print("parse_s3_location: all forms OK")

In [ ]:
def workspace_s3():
    """S3 client from short-lived MAAP workspace credentials.

    The hub's ambient identity (disasters-prod, acct 515966502221) has NO access to
    MAAP's DPS output bucket (maap-ops-workspace). Same mechanism
    shared_utils/staging_upload.py uses to publish to nasa-disasters-staging.
    """
    resp = maap.aws.workspace_bucket_credentials()
    creds = resp.get("credentials") if isinstance(resp, dict) else None
    if not isinstance(creds, dict):
        raise RuntimeError(
            "workspace_bucket_credentials() returned no 'credentials' block; "
            f"got {type(resp).__name__}"
        )
    return boto3.Session(
        aws_access_key_id=creds["aws_access_key_id"],
        aws_secret_access_key=creds["aws_secret_access_key"],
        aws_session_token=creds["aws_session_token"],
    ).client("s3")


def list_prefix(bucket, prefix, s3):
    keys = []
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
        keys.extend(o["Key"] for o in page.get("Contents", []))
    return keys


def read_s3_text(bucket, key, s3, max_bytes=2_000_000):
    return s3.get_object(Bucket=bucket, Key=key)["Body"].read(max_bytes).decode("utf-8", "replace")


if not locations:
    raise SystemExit("No locations in the result payload — see the previous cell.")

BUCKET, PREFIX = parse_s3_location(locations[0])
print(f"bucket : {BUCKET}\nprefix : {PREFIX}\n")

_s3 = workspace_s3()
keys = list_prefix(BUCKET, PREFIX, _s3)
for k in keys:
    print(" ", k)
if not keys:
    print("(empty) — the job produced no objects at this prefix.")

### Print the logs

Reads **both** files, `_stderr.txt` first — that is where your `run.sh` output lands
(see the note above). `GREP` filters to matching lines; `None` prints everything.

In [ ]:
GREP = None    # e.g. r"DPS IDENTITY|SECRETS MANAGER|ERROR"  |  None = print everything
TAIL = None    # e.g. 80 to show only the last N (post-filter) lines  |  None = all

# _stderr.txt FIRST: cwltool relays the container's console output to stderr, so
# that is where run.sh's prints are. _stdout.txt is wrapper chatter only.
for which in ("_stderr.txt", "_stdout.txt"):
    matches = [k for k in keys if k.endswith(which)]
    if not matches:
        print(f"=== {which}: not present ===\n")
        continue

    text = read_s3_text(BUCKET, matches[0], _s3)
    lines = text.splitlines()
    total = len(lines)

    if GREP:
        lines = [ln for ln in lines if re.search(GREP, ln)]
    shown = lines[-TAIL:] if TAIL else lines

    hdr = f"=== {which}  ({total} lines"
    if GREP:
        hdr += f", {len(lines)} matching"
    if TAIL and len(lines) > len(shown):
        hdr += f", last {len(shown)}"
    print(hdr + ") ===")
    print("\n".join(shown) or "(no matching lines)")
    print()

## 5. Download the whole job directory

For when you want the products, not just the log.

In [ ]:
import os

DEST = "/tmp/dps_job_download"   # local dir
DO_IT = False                    # flip to True to actually download

if DO_IT:
    os.makedirs(DEST, exist_ok=True)
    for k in keys:
        rel = os.path.relpath(k, PREFIX)
        out = os.path.join(DEST, rel)
        os.makedirs(os.path.dirname(out), exist_ok=True)
        _s3.download_file(BUCKET, k, out)
        print("->", out)
    print(f"\n{len(keys)} file(s) -> {DEST}")
else:
    print("DO_IT is False — nothing downloaded. Set it to True to fetch "
          f"{len(keys) if 'keys' in dir() else '?'} file(s) into {DEST}.")